# 3. Rolling out-of-sample forecasts and backtests (Tables 3–4, Figures 2–4)

Each model is re-estimated every `refit_every` quotations on a rolling window of `window` observations (see `config.toml`). GARCH variances are updated daily with realised past returns only. Outputs: daily predictive log scores, PIT values, 95%/99% VaR and ES.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / 'config.toml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import json
import pandas as pd
from IPython.display import display, Image
from mxntail import pipeline, reporting
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
cfg = pipeline.load_config()
RES = pipeline.results_dir()

def cached(stage, output):
    """Run a pipeline stage only if its output file is missing."""
    if not (RES / output).exists():
        print(f'Running stage {stage!r} (this can take a while)...')
        pipeline.STAGES[stage](cfg)
    else:
        print(f'Using cached results/{output}. Delete it to recompute.')


In [2]:
cached('forecasts', 'forecasts_main.csv.gz')
fc = reporting.load_forecasts('main', cfg)
fc.head()

Using cached results/forecasts_main.csv.gz. Delete it to recompute.


,variant,model,date,y,nls,pit,VaR95,ES95,VaR99,ES99,cond_sd,hit95,QL95,FZ95,hit99,QL99,FZ99
0,main,Gaussian,2001-12-18,-0.071344,0.348781,0.441557,0.930956,1.164689,1.312156,1.501703,NaN,0,0.050115,-0.048228,0,0.013835,0.280378
1,main,Gaussian,2001-12-19,0.274123,0.448701,0.681034,0.930956,1.164689,1.312156,1.501703,NaN,0,0.032842,-0.048228,0,0.010380,0.280378
2,main,Gaussian,2001-12-20,0.317044,0.487755,0.707922,0.930956,1.164689,1.312156,1.501703,NaN,0,0.030696,-0.048228,0,0.009951,0.280378
3,main,Gaussian,2001-12-21,-0.317044,0.509833,0.278845,0.930956,1.164689,1.312156,1.501703,NaN,0,0.062400,-0.048228,0,0.016292,0.280378
4,main,Gaussian,2001-12-24,-0.411467,0.623047,0.225100,0.930956,1.164689,1.312156,1.501703,NaN,0,0.067121,-0.048228,0,0.017236,0.280378


**Table 3.** Mean negative log score (NLS), VaR exceedances, Kupiec (UC) and Christoffersen (CC) p-values and the Du–Escanciano unconditional ES backtest.

In [3]:
from mxntail.evaluation import summary_table, comparison_table
s = summary_table(fc, cfg['forecast']['models'], cfg['forecast']['alphas'])
s[['model','NLS','exc95','p_uc95','p_cc95','p_ues95','exc99','p_uc99','p_cc99','p_ues99']].round(4)

,model,NLS,exc95,p_uc95,p_cc95,p_ues95,exc99,p_uc99,p_cc99,p_ues99
0,Gaussian,1.1136,311,0.9907,0.0000,0.0003,123,0.0000,0.0000,0.0000
1,Student-t,0.9867,416,0.0000,0.0000,0.0000,90,0.0009,0.0000,0.0005
2,Merton,1.0048,345,0.0532,0.0000,0.6273,54,0.2828,0.0000,0.7646
3,Kou,0.9948,346,0.0466,0.0000,0.6048,55,0.3467,0.0000,0.6727
4,GARCH-t,0.9036,406,0.0000,0.0000,0.0000,96,0.0001,0.0003,0.0001
5,GARCH-skew-t,0.8994,338,0.1240,0.0089,0.1563,74,0.1457,0.3443,0.7072
6,GARCH-Merton,0.9069,369,0.0011,0.0000,0.0019,78,0.0534,0.1014,0.1229
7,GARCH-Kou,0.9059,360,0.0056,0.0000,0.0054,73,0.1821,0.2319,0.3331


**Table 4.** Diebold–Mariano tests (HAC) against GARCH-t and Model Confidence Set p-values.

In [4]:
ev = cfg['evaluation']
dm, mcs = comparison_table(fc, cfg['forecast']['models'], 'GARCH-t', ['nls','FZ95','FZ99'],
    ev['hac_lags'], ev['mcs_size'], ev['mcs_reps'], ev['mcs_block'], seed=cfg['forecast']['seed'])
display(dm.round(4)); display(mcs.round(3))

,loss,model,benchmark,mean_diff,se,t,p
0,nls,Gaussian,GARCH-t,0.2100,0.0714,2.9409,0.0033
1,nls,Student-t,GARCH-t,0.0831,0.0131,6.3676,0.0000
2,nls,Merton,GARCH-t,0.1012,0.0226,4.4868,0.0000
3,nls,Kou,GARCH-t,0.0912,0.0175,5.2006,0.0000
4,nls,GARCH-skew-t,GARCH-t,-0.0042,0.0014,-2.9128,0.0036
5,nls,GARCH-Merton,GARCH-t,0.0033,0.0026,1.2904,0.1969
6,nls,GARCH-Kou,GARCH-t,0.0024,0.0024,0.9995,0.3175
7,FZ95,Gaussian,GARCH-t,0.2106,0.0692,3.0450,0.0023
8,FZ95,Student-t,GARCH-t,0.2260,0.0685,3.3009,0.0010
9,FZ95,Merton,GARCH-t,0.2068,0.0600,3.4465,0.0006


,nls,FZ95,FZ99
model,,,
Gaussian,0.016,0.017,0.080
Student-t,0.000,0.015,0.090
Merton,0.002,0.012,0.063
Kou,0.000,0.012,0.055
GARCH-t,0.016,0.034,0.106
GARCH-skew-t,1.000,1.000,1.000
GARCH-Merton,0.016,0.049,0.106
GARCH-Kou,0.016,0.049,0.106


In [5]:
reporting._style(); reporting.FIGS.mkdir(parents=True, exist_ok=True)
reporting.fig_var(fc); reporting.fig_parameters(cfg); reporting.fig_cumloss(fc, cfg); reporting.fig_pit(fc, cfg)
print(sorted(p.name for p in reporting.FIGS.glob('*.pdf')))

['fig_cumloss.pdf', 'fig_parameters.pdf', 'fig_pit.pdf', 'fig_returns.pdf', 'fig_var.pdf']
